In [1]:
import sys
import os 

import numpy as np 
import pandas as pd 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

sys.path.append(os.path.abspath("../.."))
from src.results_manager import ResultsManager

rm = ResultsManager("gradient_boosting")

In [2]:
from src.load_processed import load_processed_data

dataset = load_processed_data("../../data/processed/preprocessed.npz")

X_train = dataset["X_train"]
X_test = dataset["X_test"]

y_train = dataset["y_train"]
y_test = dataset["y_test"]

font_test = dataset["font_test"]
italic_test = dataset["italic_test"]
strength_test = dataset["strength_test"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Number of classes:", len(np.unique(y_train)))
print("Data type:", X_train.dtype)

Train shape: (666136, 400)
Test shape: (166534, 400)
Number of classes: 256
Data type: float32


In [ ]:
model = HistGradientBoostingClassifier(
    max_depth = 10,
    max_iter = 120,
    learning_rate = 0.1,
    random_state = 42,
    max_leaf_nodes = 31,
    verbose = 2
)

In [4]:
import time 

trained_now = False 
training_time = None

model_loaded = rm.load_model()

if model_loaded is not None: 
    model = model_loaded
    print("Successfully loaded existing model.")
else:
    print("Training model ... ")
    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    print(f"Training done in {training_time:.2f} seconds")

    # save model using results manager 
    rm.save_model(model)

    trained_now = True 

Path koju gledam: ../../results/gradient_boosting/model.pkl
Training model ... 
Binning 1.918 GB of training data: 9.197 s
Binning 0.213 GB of validation data: 0.726 s
Fitting gradient boosted rounds:
[1/120] 256 trees, 7936 leaves (31 on avg), max depth = 10, train loss: 4.35756, val loss: 4.78046, in 95.968s
[2/120] 256 trees, 7936 leaves (31 on avg), max depth = 10, train loss: 10.41026, val loss: 11.62085, in 84.873s
[3/120] 256 trees, 7936 leaves (31 on avg), max depth = 10, train loss: 22.22850, val loss: 25.13723, in 82.389s
[4/120] 256 trees, 7936 leaves (31 on avg), max depth = 10, train loss: 27.66196, val loss: 30.73599, in 82.188s
[5/120] 256 trees, 7936 leaves (31 on avg), max depth = 10, train loss: 42.87672, val loss: 46.95429, in 81.176s
[6/120] 256 trees, 7936 leaves (31 on avg), max depth = 10, train loss: 57.41117, val loss: 62.58331, in 80.067s
[7/120] 256 trees, 7936 leaves (31 on avg), max depth = 10, train loss: 69.98175, val loss: 81.05326, in 79.102s
[8/120] 25

In [5]:

y_pred = model.predict(X_test)

In [6]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average = "macro")
weighted_f1 = f1_score(y_test, y_pred, average = "weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.43455390490830703
Macro F1: 0.29455074435722994
Weighted F1: 0.4344363952170593


In [7]:
import json 

if training_time is None:
    training_time = -0.1

results = {
    "model": "gradient_boosting",
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "training_time_seconds": float(training_time)
}

# save results in .json file only if model is trained now: 
if trained_now:
    rm.save_metrics(results)


Metrics saved to ../../results/gradient_boosting/metrics.json


In [8]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(f"Shape: {cm.shape}\n")
print(cm)

Shape: (256, 256)

[[ 77   2   1 ...   1   0   0]
 [  7  81   8 ...   0   0   0]
 [  1  27 174 ...   1   1   2]
 ...
 [  0   1   2 ... 107   3   3]
 [  2   2   3 ...   1  94  15]
 [  0   0   0 ...   0   6  50]]


In [9]:
errors = []

n = cm.shape[0]

for true_class in range(n):
    for pred_class in range(n):
        if true_class != pred_class:            # avoid diag elements 
            count = cm[true_class, pred_class]
            errors.append((true_class, pred_class, count))

errors = sorted(errors, key = lambda x: x[2], reverse = True)

top_errors = errors[:10]

top_errors_readable = []
for true_class, pred_class, count in top_errors:
    label = f"{chr(true_class)} → {chr(pred_class)}"
    top_errors_readable.append((label, count))

errors_df = pd.DataFrame(
    top_errors_readable,
    columns = ["confusion", "count"]
)

rm.save_dataframe(errors_df, "top_confusions")

errors_df

Dataframe saved to ../../results/gradient_boosting/top_confusions.csv


,confusion,count
0,O → 0,146
1,9 → 4,143
2,/ → 1,126
3,| → _,123
4,2 → 1,119
5,5 → 6,105
6, → _,103
7,8 → 1,103
8,6 → 1,102
9,I → 1,100


### Per-class accuracy 

In [10]:
# number of correct classifications per class (diag elements)
correct = np.diag(cm)

# total number of samples per class
total = cm.sum(axis = 1)

# per-class accuracy
per_class_accuracy = correct / total

# ASCII oznake klasa
labels = [f"{i}:{repr(chr(i))}" for i in range(cm.shape[0])]

# dataframe sa rezultatima
per_class_df = pd.DataFrame({
    "character": labels,
    "accuracy": per_class_accuracy,
    "correct": correct,
    "total": total,
    "percentage_%": per_class_accuracy *100
})

# sortiranje po tacnosti
per_class_df = per_class_df.sort_values("accuracy")
rm.save_dataframe(per_class_df, "class_accuracy")

worst_classified = per_class_df.head(10)
rm.save_dataframe(worst_classified, "worst_classified_classes")
print("Worst classified characters:")
print(worst_classified)

best_classified = per_class_df.tail(10)
rm.save_dataframe(best_classified, "best_classified_classes")
print("\nBest classified characters:")
print(best_classified)

Dataframe saved to ../../results/gradient_boosting/class_accuracy.csv
Dataframe saved to ../../results/gradient_boosting/worst_classified_classes.csv
Worst classified characters:
      character  accuracy  correct  total  percentage_%
8      8:'\x08'  0.071698       19    265      7.169811
156  156:'\x9c'  0.086022       16    186      8.602151
173  173:'\xad'  0.087065       35    402      8.706468
18    18:'\x12'  0.087179       34    390      8.717949
3      3:'\x03'  0.087227       28    321      8.722741
175     175:'¯'  0.090206       35    388      9.020619
151  151:'\x97'  0.092784       18    194      9.278351
135  135:'\x87'  0.095455       21    220      9.545455
157  157:'\x9d'  0.109890       20    182     10.989011
133  133:'\x85'  0.115079       29    252     11.507937
Dataframe saved to ../../results/gradient_boosting/best_classified_classes.csv

Best classified characters:
   character  accuracy  correct  total  percentage_%
57    57:'9'  0.662993     2587   3902     6

In [11]:
import warnings
warnings.filterwarnings("ignore")

font_results = []

for font in np.unique(font_test):
    mask = font_test == font

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    font_results.append({
        "font": font,
        "samples": mask.sum(),
        "accuracy": acc
    })


font_df = pd.DataFrame(font_results)
font_df = font_df.sort_values("accuracy", ascending = False)
font_df["accuracy_%"] = (font_df["accuracy"] * 100).round(2)
rm.save_dataframe(font_df, "font_accuracy")

font_df

Dataframe saved to ../../results/gradient_boosting/font_accuracy.csv


,font,samples,accuracy,accuracy_%
43,E13B,4718,0.854387,85.44
146,VIN,218,0.853211,85.32
68,HANDPRINT,13998,0.784826,78.48
102,OCRB,18706,0.747568,74.76
101,OCRA,13077,0.720808,72.08
...,...,...,...,...
123,SCRIPT,438,0.093607,9.36
82,KUNSTLER,173,0.086705,8.67
152,YI BAITI,1206,0.077114,7.71
45,EDWARDIAN,178,0.067416,6.74


In [12]:
italic_results = []

for value in np.unique(italic_test):
    mask = italic_test == value 

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    italic_results.append({
        "italic": value, 
        "samples": mask.sum(),
        "accuracy": acc
    })

italic_df = pd.DataFrame(italic_results)
italic_df["accuracy_%"] = (italic_df["accuracy"] * 100).round(2)
rm.save_dataframe(italic_df, "italic_accuracy")

italic_df

Dataframe saved to ../../results/gradient_boosting/italic_accuracy.csv


,italic,samples,accuracy,accuracy_%
0,0,116206,0.515335,51.53
1,1,50328,0.248033,24.80


57.13% karaktera koji nisu italic je klasifikovano tacno  
26.15% karaktera koji jesu italic je klasifikovano tacno

In [13]:
bins = np.linspace(0, 1, 6)   # 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

strength_results = []

for i in range(len(bins)-1):

    mask = (strength_test >= bins[i]) & (strength_test < bins[i+1])

    samples = mask.sum()
    if samples == 0:
        acc = np.nan
    else: 
        acc = accuracy_score(
            y_test[mask],
            y_pred[mask]
        )


    strength_results.append({
        "strength_range": labels[i],
        "samples": samples,
        "accuracy": acc
    })


strength_df = pd.DataFrame(strength_results)
strength_df["accuracy_%"] = (strength_df["accuracy"] * 100).round(2)

rm.save_dataframe(strength_df, "strength_accuracy")

strength_df

Dataframe saved to ../../results/gradient_boosting/strength_accuracy.csv


,strength_range,samples,accuracy,accuracy_%
0,0.0-0.2,0,NaN,NaN
1,0.2-0.4,0,NaN,NaN
2,0.4-0.6,114924,0.518821,51.88
3,0.6-0.8,51610,0.246910,24.69
4,0.8-1.0,0,NaN,NaN


Deblji karakteri imaju više spojenih piksela, pa se oblici slova međusobno približavaju.  
Na primer: 8 vs B, 5 vs S, O vs 0 -> jos slicniji kada je font deblji